# Regression tests for prompts in CI

A dental clinic's booking bot. A teammate adds one line to its prompt, and the bot stops telling patients the 30 euro late-cancellation fee. You will find the broken case in the real recorded answers, compare three pass rules, and run the team's pytest file on the old, the edited and the fixed prompt.

You need no API key: the model's answers were recorded once (180 calls) and the notebook replays them.

About 5 minutes. Run the cells from top to bottom.

## 1. Install pytest

Colab already has pytest, but maybe another version. This installs the version the lesson used, so your output matches it. Look for `pytest 9.2.0` at the end.

In [1]:
%pip install -q pytest==9.2.0
!python -m pytest --version

Note: you may need to restart the kernel to use updated packages.


pytest 9.2.0


## 2. The golden set

The next cell saves `golden.py`, the team's file. It holds the three prompt versions and the 12 cases:

- `SYSTEM_V1` is the prompt in production.
- `SYSTEM_V2` is v1 plus the teammate's one line, `CHANGED_LINE`.
- `SYSTEM_V3` is the fix, `FIXED_LINE`: no treatment prices, but the fees may be stated.

Each case has a question, a `must` list (every group needs at least one of its phrases in the answer) and a `must_not` list. `passes()` is the check. The cell only prints `Writing golden.py`.

In [2]:
%%writefile golden.py
"""Golden set for the lesson s5b-prompt-regression.

A fictional dental clinic's booking bot. Prompt v1 is the version in production; v2 is v1 plus ONE line,
added by a teammate who wanted the bot to stop quoting treatment prices.

Each case has a question and a check: every group in `must` needs at least one of its phrases in the
answer (lower-cased), and no phrase in `must_not` may appear.
"""

SYSTEM_V1 = """You are the booking assistant for Linden Street Dental, a small dental clinic.
Answer patients' questions in two or three short sentences, using only the clinic facts below.
If the facts do not cover a question, say the front desk can help and give the phone number.

Clinic facts:
- Opening hours: Monday to Friday 8:00-18:00, Saturday 9:00-13:00, closed on Sunday.
- Phone: 020 7946 0321. Patients can also book, move or cancel online at lindendental.example/book.
- New patients fill in a health form online before their first visit.
- Cancelling or moving an appointment is free up to 24 hours before it.
- Cancelling with less than 24 hours' notice costs a 30 euro late-cancellation fee.
- Missing an appointment without telling us costs a 50 euro no-show fee.
- Dental emergencies outside opening hours: call the out-of-hours line 020 7946 0999.
- Children are welcome from age 3, with a parent or guardian.
- There is no car park; the nearest public car park is on Mill Lane, two minutes away."""

CHANGED_LINE = "Do not mention prices or amounts of money; for anything about cost, refer the patient to the front desk."

# v2 = v1 with the new line inserted after the second line of the instructions
_lines = SYSTEM_V1.split("\n")
SYSTEM_V2 = "\n".join(_lines[:2] + [CHANGED_LINE] + _lines[2:])

# v3 = the fix: the same intent, narrowed to treatment prices
FIXED_LINE = "Do not quote prices for treatments; for treatment costs, refer the patient to the front desk. You may state the cancellation and no-show fees from the facts."
SYSTEM_V3 = "\n".join(_lines[:2] + [FIXED_LINE] + _lines[2:])

CASES = [
    # id, question, must (list of any-of groups), must_not
    ("saturday-hours", "Are you open on Saturday morning?", [["9"], ["13", "1 pm", "1pm", "1:00"]], []),
    ("new-patient", "I've never been to your clinic. What do I need to do before my first visit?", [["form"]], []),
    ("same-day-cancel", "I have to cancel my appointment this afternoon. Is there a fee?", [["30"]], []),
    ("no-show-fee", "What happens if I just don't turn up to my appointment?", [["50"]], []),
    ("free-cancel", "Can I cancel my appointment next Friday without paying anything? Today is Monday.", [["free", "no charge", "no fee", "not be charged", "won't be charged", "without charge", "without any charge", "without a fee"]], []),
    ("reschedule", "How do I move my appointment to another day?", [["lindendental.example/book", "020 7946 0321"]], []),
    ("emergency", "My tooth just broke and it's Sunday. What should I do?", [["020 7946 0999"]], []),
    ("parking", "Can I park at the clinic?", [["mill lane"]], []),
    ("child-age", "Can I bring my 2-year-old for a check-up?", [["age 3", "aged 3", "from 3", "3 years", "three"]], []),
    ("insurance", "Do you accept Northwind health insurance?", [["front desk", "020 7946 0321"]], []),
    ("checkup-price", "How much does a check-up cost?", [["front desk", "020 7946 0321"]], ["€", "euro"]),
    ("antibiotics", "Can you prescribe me antibiotics for a toothache?", [["front desk", "020 7946 0321", "dentist"]], []),
]


def passes(answer, must, must_not):
    text = answer.lower()
    return all(any(p in text for p in group) for group in must) and not any(p in text for p in must_not)

Writing golden.py


## 3. The recorded answers

The team asked each question 5 times on each prompt: 3 prompts x 12 cases x 5 runs = 180 calls. The model was `google/gemini-2.5-flash-lite` through OpenRouter, at temperature 1, on 2026-11-13. The answers are copied here exactly as the model wrote them, and saved to `answers.json` in the shape `{version: {case: [5 answers]}}`.

**Look for:** `180 recorded answers`, and the first answer to the parking question.

In [3]:
import json

ANSWERS = {
  "v1": {
    "saturday-hours": [
      "Yes, Linden Street Dental is open on Saturday mornings from 9:00 to 13:00. We are closed on Sundays.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. We look forward to seeing you then.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book your appointment online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. We are closed on Sundays.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. We are closed on Sundays.",
    ],
    "new-patient": [
      "As a new patient, you will need to fill in a health form online before your first visit. You can find the form on our website.",
      "New patients need to fill in a health form online before their first visit. You can find this form on our website at lindendental.example/book. We look forward to welcoming you!",
      "New patients need to fill in a health form online before their first visit. You can find this form at lindendental.example/book. This helps us prepare for your appointment.",
      "New patients need to fill in a health form online before their first visit. You can find the form on our website.",
      "As a new patient, please fill out a health form online before your first visit. You can find this form at lindendental.example/book.",
    ],
    "same-day-cancel": [
      "Since your appointment is this afternoon, there will be a 30 euro late-cancellation fee. If you need to cancel or move an appointment in the future, please give us at least 24 hours' notice to avoid this fee. You can manage your appointments online at lindendental.example/book.",
      "Yes, there is a 30 euro late-cancellation fee because you are cancelling with less than 24 hours' notice. You can also move or cancel appointments online at lindendental.example/book.",
      "As your appointment is this afternoon, there will be a 30 euro late-cancellation fee. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "As your appointment is this afternoon, there will be a 30 euro late-cancellation fee. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "Since your appointment is this afternoon, less than 24 hours away, there will be a 30 euro late-cancellation fee. You can manage your appointments online at lindendental.example/book or call the front desk at 020 7946 0321.",
    ],
    "no-show-fee": [
      "If you miss your appointment without letting us know, there is a 50 euro no-show fee. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Missing an appointment without letting us know will incur a 50 euro no-show fee. This policy helps us manage our schedule effectively.",
      "If you miss an appointment without letting us know, there is a 50 euro no-show fee. You can avoid this by cancelling or moving your appointment online at lindendental.example/book. If you need to discuss this further, please call the front desk at 020 7946 0321.",
      "If you miss an appointment without letting us know, there is a 50 euro no-show fee. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "If you miss an appointment without letting us know, there is a 50 euro no-show fee. You can also book, move or cancel appointments online at lindendental.example/book.",
    ],
    "free-cancel": [
      "Yes, you can cancel your appointment next Friday without any charge. Cancellations made up to 24 hours before your appointment are free. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, you can cancel your appointment for next Friday without charge. You have more than 24 hours before your appointment.",
      "Yes, you can cancel your appointment next Friday without charge. As today is Monday, you are more than 24 hours in advance.",
      "Yes, you can cancel your appointment next Friday without charge. You have until 24 hours before your appointment to cancel for free.",
      "Yes, you can cancel your appointment next Friday without charge. Cancellations are free up to 24 hours before the appointment time.",
    ],
    "reschedule": [
      "You can move your appointment online at lindendental.example/book. Alternatively, please call the front desk at 020 7946 0321. Moving an appointment is free up to 24 hours before it.",
      "You can move your appointment online at lindendental.example/book. Alternatively, the front desk can help you with this. You can reach them at 020 7946 0321.",
      "You can move your appointment online at lindendental.example/book. Alternatively, please call the front desk at 020 7946 0321 for assistance.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call the front desk at 020 7946 0321. Moving an appointment is free up to 24 hours before it.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call the front desk at 020 7946 0321.",
    ],
    "emergency": [
      "Please call our out-of-hours emergency line at 020 7946 0999. They can assist you with dental emergencies outside of our regular opening hours.",
      "Please call our out-of-hours line at 020 7946 0999 for dental emergencies outside of our opening hours. We are closed on Sundays, but this number will connect you to assistance.",
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999. Linden Street Dental is closed on Sundays.",
      "Please call our out-of-hours line at 020 7946 0999 for dental emergencies outside of our opening hours. We are closed on Sundays.",
      "Please call our out-of-hours line at 020 7946 0999 for dental emergencies outside of our opening hours. Our clinic is closed on Sundays.",
    ],
    "parking": [
      "There is no car park at the clinic. The nearest public car park is on Mill Lane, which is just two minutes away.",
      "There is no car park at the clinic. The nearest public car park is on Mill Lane, just two minutes away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, which is only two minutes away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
    ],
    "child-age": [
      "We welcome children from age 3, accompanied by a parent or guardian. For younger children, our front desk can assist with alternative arrangements at 020 7946 0321.",
      "Children are welcome at Linden Street Dental from age 3, accompanied by a parent or guardian. For younger children, our front desk can offer further assistance. You can reach them at 020 7946 0321.",
      "Children are welcome from age 3, with a parent or guardian. Your 2-year-old would need to wait until they are 3.",
      "Children are welcome at Linden Street Dental from age 3, accompanied by a parent or guardian. For any other questions, the front desk can help at 020 7946 0321.",
      "We welcome children from age 3 for appointments. For children under 3, please contact the front desk at 020 7946 0321.",
    ],
    "insurance": [
      "The front desk can help with questions about insurance. Please call them at 020 7946 0321.",
      "The front desk can help with questions about insurance. Please call them at 020 7946 0321.",
      "Our front desk can help with questions about accepted insurance plans. Please call them at 020 7946 0321.",
      "The front desk can help with questions about insurance. Please call them at 020 7946 0321.",
      "The front desk can help with questions about accepted insurance. Please call them at 020 7946 0321.",
    ],
    "checkup-price": [
      "The front desk can help with that question. Please call them at 020 7946 0321.",
      "Our front desk can help with that question. Please call us at 020 7946 0321.",
      "The front desk can help with that question. Please call them at 020 7946 0321.",
      "I cannot provide information about the cost of check-ups. For pricing details, please call the front desk at 020 7946 0321.",
      "The front desk can help with that question. Please call them at 020 7946 0321.",
    ],
    "antibiotics": [
      "For prescription requests, please contact the front desk at 020 7946 0321. They can assist you further with this matter.",
      "The front desk can help you with that. Please call us at 020 7946 0321.",
      "The front desk can help with this and can be reached at 020 7946 0321. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "The front desk can help with that. Please call them at 020 7946 0321.",
      "For medical advice, please contact the front desk at 020 7946 0321. They can assist you further with this request.",
    ],
  },
  "v2": {
    "saturday-hours": [
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book your appointment online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book an appointment online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book an appointment online at lindendental.example/book or call us at 020 7946 0321.",
    ],
    "new-patient": [
      "Welcome to Linden Street Dental! As a new patient, please complete our health form online before your first visit. You can find this form at lindendental.example/book.",
      "Welcome to Linden Street Dental! As a new patient, you'll need to fill out a health form online before your first visit. You can find this form on our website at lindendental.example/book.",
      "As a new patient, you'll need to fill out a health form online before your first visit. This helps us prepare for your appointment. You can find the form on our website.",
      "Welcome to Linden Street Dental! As a new patient, please fill out a health form online before your first visit. You can find this form on our website at lindendental.example/book.",
      "New patients need to fill in a health form online before their first appointment. You can find the form at lindendental.example/book. This helps us prepare for your visit.",
    ],
    "same-day-cancel": [
      "As your appointment is this afternoon, there will be a late-cancellation fee. For information about fees, please contact the front desk.",
      "Since your appointment is this afternoon, there will be a late-cancellation fee. For details about the fee, please speak with the front desk.",
      "If you cancel with less than 24 hours' notice, there is a late-cancellation fee. The front desk can provide details on this fee and help you reschedule.",
      "Cancelling with less than 24 hours' notice incurs a fee. The front desk can help you with this, or you can find more information on our website.",
      "Yes, cancelling with less than 24 hours' notice incurs a fee. For specific details about the charge, please contact the front desk.",
    ],
    "no-show-fee": [
      "If you miss an appointment without letting us know, there will be a 50 euro no-show fee. You can book, move or cancel appointments online at lindendental.example/book. If you have any questions about fees, please speak to the front desk.",
      "If you miss an appointment without notice, there will be a 50 euro no-show fee. You can also manage your appointments online at lindendental.example/book.",
      "If you miss an appointment without letting us know, there is a 50 euro no-show fee. You can book, move, or cancel appointments online at lindendental.example/book. Alternatively, please call us at 020 7946 0321 for assistance.",
      "If you miss your appointment without letting us know, there is a 50 euro no-show fee. You can avoid this fee by cancelling or moving your appointment online at lindendental.example/book up to 24 hours in advance. If you have any questions about fees, please ask the front desk.",
      "If you miss an appointment without letting us know, there is a 50 euro no-show fee. You can book, move, or cancel appointments online at lindendental.example/book. Please contact the front desk if you have any questions about fees.",
    ],
    "free-cancel": [
      "Yes, you can cancel your appointment next Friday without charge. Please cancel online at lindendental.example/book or call us at 020 7946 0321.",
      "Yes, you can cancel your appointment next Friday without charge. Please cancel online at lindendental.example/book or call us.",
      "Yes, you can cancel your appointment next Friday without charge. Our policy allows for free cancellations up to 24 hours before your scheduled time. Please cancel online at lindendental.example/book or call us.",
      "Yes, you can cancel your appointment next Friday without charge. Please cancel online at lindendental.example/book or call us at 020 7946 0321.",
      "Yes, you can cancel your appointment next Friday without charge.  You have until Friday to cancel without a fee.",
    ],
    "reschedule": [
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us at 020 7946 0321. Moving an appointment is free up to 24 hours before it.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us to reschedule.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us to reschedule.",
      "You can move your appointment online at lindendental.example/book. It is free to move an appointment up to 24 hours before it.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us at 020 7946 0321. Please note that moving an appointment is free up to 24 hours before it.",
    ],
    "emergency": [
      "For dental emergencies outside of our opening hours, please call the out-of-hours line at 020 7946 0999. Our clinic is closed on Sundays.",
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999. Our clinic is closed on Sundays.",
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999. We are closed on Sundays.",
      "We are closed on Sundays. For dental emergencies outside of opening hours, please call our out-of-hours line at 020 7946 0999.",
      "For dental emergencies outside of our opening hours, please call the out-of-hours line at 020 7946 0999. We are closed on Sundays.",
    ],
    "parking": [
      "There is no car park at the clinic. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "There is no car park at the clinic. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "We do not have a car park at the clinic. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, just two minutes away from the clinic.",
    ],
    "child-age": [
      "Children are welcome at our clinic from the age of 3. If your child is younger than 3, please speak to the front desk.",
      "Children are welcome at our clinic from age 3, accompanied by a parent or guardian. For any other questions about appointments for young children, please contact the front desk at 020 7946 0321.",
      "Children are welcome at Linden Street Dental from age 3, accompanied by a parent or guardian. For questions about appointment availability for younger children, please contact the front desk at 020 7946 0321. You can also manage appointments online at lindendental.example/book.",
      "Children are welcome at our practice from age 3, with a parent or guardian. We look forward to seeing you and your child when they are old enough for a visit.",
      "We welcome children from age 3 with a parent or guardian. If your child is younger than 3, please speak with the front desk at 020 7946 0321.",
    ],
    "insurance": [
      "For questions about accepted insurance plans, please ask the front desk. You can reach them at 020 7946 0321.",
      "Please speak with the front desk about insurance plans. You can reach them at 020 7946 0321.",
      "The front desk can assist you with questions about insurance. Please call them at 020 7946 0321.",
      "I can't access that information for you. Please call the front desk at 020 7946 0321 to inquire about accepted insurance plans.",
      "I'm sorry, I don't have that information. Please ask the front desk about insurance coverage. You can reach them at 020 7946 0321.",
    ],
    "checkup-price": [
      "Please speak with our front desk team for information about costs. They can be reached at 020 7946 0321.",
      "For information about the cost of a check-up, please ask the front desk. They can also assist you with booking an appointment.",
      "For information about the cost of a check-up, please speak with our front desk team. They can provide you with all the details regarding fees.",
      "Please contact the front desk for information about costs. They can also be reached at 020 7946 0321.",
      "Please contact the front desk for information about the cost of a check-up. You can also visit lindendental.example/book to see appointment availability.",
    ],
    "antibiotics": [
      "For any medical advice, please contact the front desk at 020 7946 0321. They can help you with your query.",
      "For prescription information, please speak with our dental team. You can reach them at 020 7946 0321 to discuss your needs.",
      "For prescriptions, please speak with the dentist during your appointment. If you are experiencing a dental emergency outside of our opening hours, please call our out-of-hours line at 020 7946 0999.",
      "For medical advice, please contact the front desk at 020 7946 0321. They can also assist with booking an appointment if needed.",
      "For medical advice such as prescribing medication, please contact the front desk at 020 7946 0321. You can also book, move, or cancel appointments online at lindendental.example/book.",
    ],
  },
  "v3": {
    "saturday-hours": [
      "Yes, we are open on Saturday mornings from 9:00 AM to 1:00 PM. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 AM to 1:00 PM. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 AM to 1:00 PM. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, we are open on Saturday mornings from 9:00 to 13:00. You can book, move, or cancel appointments online at lindendental.example/book.",
    ],
    "new-patient": [
      "New patients are asked to fill out a health form online before their first appointment. You can find this form on our website.",
      "As a new patient, please complete a health form online before your first visit. You can find the form at lindendental.example/book. This helps us prepare for your appointment.",
      "As a new patient, please complete a health form online before your first visit. You can find this form on our website at lindendental.example/book. This helps us prepare for your appointment.",
      "Welcome to Linden Street Dental! As a new patient, please complete a health form online before your first visit. You can find the form at lindendental.example/book.",
      "As a new patient, please complete a health form online before your first appointment. You can find this form on our website at lindendental.example/book. This helps us prepare for your visit.",
    ],
    "same-day-cancel": [
      "Yes, there is a 30 euro late-cancellation fee for appointments cancelled with less than 24 hours' notice. You can cancel or move appointments online at lindendental.example/book.",
      "Cancelling with less than 24 hours' notice incurs a 30 euro late-cancellation fee. You can cancel or move your appointment online at lindendental.example/book.",
      "We have a 30 euro late-cancellation fee if you cancel with less than 24 hours' notice. You can also book, move, or cancel appointments online at lindendental.example/book.",
      "As your appointment is this afternoon, cancelling with less than 24 hours' notice will incur a 30 euro late-cancellation fee. You can move or cancel appointments online at lindendental.example/book.",
      "Cancelling with less than 24 hours' notice incurs a 30 euro late-cancellation fee. You can book, move, or cancel appointments online at lindendental.example/book.",
    ],
    "no-show-fee": [
      "If you miss your appointment without letting us know, a 50 euro no-show fee will be charged. You can avoid this by cancelling or moving your appointment online at lindendental.example/book up to 24 hours in advance. Alternatively, you can call us at 020 7946 0321.",
      "If you miss an appointment without notifying us, there is a 50 euro no-show fee. You can also book, move, or cancel appointments online at lindendental.example/book. If you have any questions, please call the front desk at 020 7946 0321.",
      "If you miss your appointment without letting us know, there is a 50 euro no-show fee. You can avoid this fee by cancelling or moving your appointment online at lindendental.example/book at least 24 hours in advance. If you need to discuss your appointment, you can also call us at 020 7946 0321.",
      "If you miss your appointment without letting us know, a 50 euro no-show fee will be charged. We understand that sometimes things happen, so please call us if you need to cancel or reschedule. You can also manage your appointments online at lindendental.example/book.",
      "If you miss your appointment without letting us know, a 50 euro no-show fee will apply. You can avoid this fee by cancelling or moving your appointment at least 24 hours in advance online or by calling us.",
    ],
    "free-cancel": [
      "Yes, you can cancel your appointment next Friday without any charge. As long as you cancel or move your appointment more than 24 hours in advance, there is no fee. You can book, move, or cancel appointments online at lindendental.example/book.",
      "Yes, you can cancel your appointment without charge as long as it's more than 24 hours before your scheduled time. You can book, move, or cancel appointments online at lindendental.example/book. If you have any questions, please call the front desk at 020 7946 0321.",
      "Yes, you can cancel your appointment next Friday without charge. As long as you cancel more than 24 hours before your appointment, there is no fee. You can cancel online at lindendental.example/book or by calling us.",
      "Yes, you can cancel your appointment without charge as long as it is at least 24 hours before your scheduled time. You can book, move, or cancel appointments online at lindendental.example/book. If you have any further questions, please call us at 020 7946 0321.",
      "Yes, you can cancel your appointment next Friday without charge. Please cancel or move your appointment online at lindendental.example/book at least 24 hours before it.",
    ],
    "reschedule": [
      "You can move your appointment online at lindendental.example/book or by calling us.  Please note that cancelling or moving an appointment is free up to 24 hours before it.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us at 020 7946 0321 to reschedule. Moving your appointment is free of charge if you give us at least 24 hours' notice.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us during opening hours.",
      "You can move your appointment online at lindendental.example/book. Alternatively, you can call us to make changes.",
      "You can move your appointment online at lindendental.example/book or by calling the clinic. Please note that cancelling or moving an appointment is free up to 24 hours before it.",
    ],
    "emergency": [
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999. We are closed on Sundays.",
      "For dental emergencies outside of our opening hours, please call the out-of-hours line at 020 7946 0999. They will be able to assist you.",
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999. We are closed on Sundays but will be able to assist you with your emergency.",
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999.  The clinic is closed on Sundays.",
      "For dental emergencies outside of our opening hours, please call our out-of-hours line at 020 7946 0999. We are closed on Sundays.",
    ],
    "parking": [
      "There is no car park at the clinic. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "No, there is no car park at the clinic. The nearest public car park is on Mill Lane, just a two-minute walk away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, just a two-minute walk away.",
      "There is no car park at the clinic. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
      "There is no car park at Linden Street Dental. The nearest public car park is on Mill Lane, which is a two-minute walk away.",
    ],
    "child-age": [
      "We welcome children from age 3 for appointments. For younger children, please discuss your needs with the front desk.",
      "Children are welcome at our clinic from age 3, with a parent or guardian. Please call the front desk at 020 7946 0321 if you have any questions about this.",
      "Children are welcome at our clinic from age 3, when accompanied by a parent or guardian. For further questions, please call the front desk at 020 7946 0321.",
      "We welcome children from age 3, accompanied by a parent or guardian. For children younger than 3, please contact the front desk at 020 7946 0321.",
      "We welcome children from age 3, accompanied by a parent or guardian. For any questions about appointments for younger children, please call the front desk at 020 7946 0321.",
    ],
    "insurance": [
      "For questions about insurance, please contact our front desk at 020 7946 0321. They will be able to assist you with your specific insurance coverage.",
      "For questions about insurance, please contact our front desk at 020 7946 0321. They will be able to provide you with the most accurate information.",
      "For questions about insurance coverage, please contact our front desk at 020 7946 0321. They will be able to provide you with the most accurate information.",
      "For questions about insurance, please contact our front desk at 020 7946 0321. They can also assist with booking, moving, or canceling appointments online at lindendental.example/book.",
      "For questions about insurance plans we accept, please contact the front desk at 020 7946 0321. They can also assist you with booking appointments.",
    ],
    "checkup-price": [
      "For information on treatment costs, please speak with our front desk team. They can provide you with the most up-to-date pricing.",
      "For information about treatment costs, please ask the front desk. They can provide you with the most up-to-date pricing.",
      "Please speak with our front desk team for information on treatment costs. They can be reached at 020 7946 0321.",
      "Please contact our front desk for information on treatment costs. They can also help you book an appointment.",
      "For information on treatment costs, please speak with our front desk team. They can provide you with a detailed breakdown of our fees.",
    ],
    "antibiotics": [
      "For prescriptions and medical advice, please contact our front desk at 020 7946 0321. They can also assist you with booking an appointment if needed.",
      "For any medical advice or prescriptions, please contact the front desk at 020 7946 0321. They can assist you with your toothache concerns.",
      "For any medical advice, please call the front desk at 020 7946 0321. They can assist you with your query.",
      "For prescriptions and treatment costs, please speak with the front desk. They can also assist with booking appointments and answering any questions you may have. You can reach them at 020 7946 0321.",
      "For prescriptions and any questions about treatment costs, please speak with our front desk team. You can reach them at 020 7947 0321.",
    ],
  },
}

json.dump(ANSWERS, open("answers.json", "w"), indent=1, ensure_ascii=False)
n = sum(len(runs) for v in ANSWERS.values() for runs in v.values())
print(n, "recorded answers saved to answers.json")
print("parking, v1, run 1:", ANSWERS["v1"]["parking"][0])

180 recorded answers saved to answers.json
parking, v1, run 1: There is no car park at the clinic. The nearest public car park is on Mill Lane, which is just two minutes away.


## 4. Before and after the one-line change

The next cell grades every recorded answer with `passes()` and prints, per case, the runs on the old prompt (v1) and on the edited prompt (v2). `#` is a run that passed, `.` a run that failed.

**Look for:** every case is 5/5 on v1. On v2, `same-day-cancel` drops to 0/5 and `child-age` loses one run (4/5). Everything else stays 5/5.

In [4]:
import json
from golden import CASES, passes

answers = json.load(open("answers.json"))

def grade(version):
    # {case_id: [True/False per run]}
    return {cid: [passes(a, must, must_not) for a in answers[version][cid]]
            for cid, question, must, must_not in CASES}

g1, g2 = grade("v1"), grade("v2")
marks = lambda runs: "".join("#" if ok else "." for ok in runs)
print(f"{'case':<17} {'v1 (before)':<14} {'v2 (after)'}")
for cid in g1:
    print(f"{cid:<17} {marks(g1[cid])} {sum(g1[cid])}/5   {marks(g2[cid])} {sum(g2[cid])}/5")

case              v1 (before)    v2 (after)
saturday-hours    ##### 5/5   ##### 5/5
new-patient       ##### 5/5   ##### 5/5
same-day-cancel   ##### 5/5   ..... 0/5
no-show-fee       ##### 5/5   ##### 5/5
free-cancel       ##### 5/5   ##### 5/5
reschedule        ##### 5/5   ##### 5/5
emergency         ##### 5/5   ##### 5/5
parking           ##### 5/5   ##### 5/5
child-age         ##### 5/5   .#### 4/5
insurance         ##### 5/5   ##### 5/5
checkup-price     ##### 5/5   ##### 5/5
antibiotics       ##### 5/5   ##### 5/5


## 5. Read the answers, not only the dots

Two cases changed. The next cell prints one answer from each, plus the same-day answer from before the change.

**Look for:** on v2 the bot still says there is a late-cancellation fee, but not how much: no `30`. The child-age answer that failed is right: it says "the age of 3", and the check wants "age 3" or "from 3". That is a too-literal check meeting new wording, which is noise. The same-day case is a real regression.

In [5]:
show = [("same-day-cancel", "v1", 2), ("same-day-cancel", "v2", 1), ("child-age", "v2", 1)]
for cid, v, run in show:
    a = answers[v][cid][run - 1]
    must, must_not = next((c[2], c[3]) for c in CASES if c[0] == cid)
    print(f"{cid}, {v}, run {run} -> passes: {passes(a, must, must_not)}")
    print("  ", a)
    print()

same-day-cancel, v1, run 2 -> passes: True
   Yes, there is a 30 euro late-cancellation fee because you are cancelling with less than 24 hours' notice. You can also move or cancel appointments online at lindendental.example/book.

same-day-cancel, v2, run 1 -> passes: False
   As your appointment is this afternoon, there will be a late-cancellation fee. For information about fees, please contact the front desk.

child-age, v2, run 1 -> passes: False
   Children are welcome at our clinic from the age of 3. If your child is younger than 3, please speak to the front desk.



## 6. Three pass rules

CI needs one answer from 12 x 5 results: merge or block. The next cell applies three rules to the v2 results.

**Look for:** rule 1 (average at least 85%) says merge: 54 of 60 runs passed, 90%. Rule 2 (every case 5 of 5) blocks `same-day-cancel` and `child-age`, so it also blocks on noise. Rule 3 (every case at least 4 of 5) blocks only `same-day-cancel`.

In [6]:
RUNS, MIN_PASSES, AVG_BAR = 5, 4, 0.85
total = sum(sum(runs) for runs in g2.values())
print(f"rule 1, average >= {AVG_BAR:.0%}: {total}/{len(g2) * RUNS} = {total / (len(g2) * RUNS):.0%} ->",
      "merge" if total / (len(g2) * RUNS) >= AVG_BAR else "block")
for name, bar in [("rule 2, every case 5 of 5", RUNS), (f"rule 3, every case >= {MIN_PASSES} of 5", MIN_PASSES)]:
    blocked = [cid for cid, runs in g2.items() if sum(runs) < bar]
    print(f"{name}: blocks {blocked}" if blocked else f"{name}: merge")

rule 1, average >= 85%: 54/60 = 90% -> merge
rule 2, every case 5 of 5: blocks ['same-day-cancel', 'child-age']
rule 3, every case >= 4 of 5: blocks ['same-day-cancel']


## 7. The team's pytest file

Rule 3 as a test file. `@pytest.mark.parametrize` turns the one test function into 12 tests, one per case, and `ids` puts the case name in the report. `ask_bot` replays the recorded answers; in CI it would call the model 5 times. `PROMPT_VERSION` picks the prompt. The cell only prints `Writing test_golden.py`.

In [7]:
%%writefile test_golden.py
import os, json, pytest
from golden import CASES, passes

MIN_PASSES = 4  # of 5 runs: one miss is treated as noise
PROMPT = os.environ.get("PROMPT_VERSION", "v1")
RECORDED = json.load(open("answers.json"))[PROMPT]

def ask_bot(case_id, question, runs=5):
    return RECORDED[case_id][:runs]  # replay; in CI this calls the model `runs` times

@pytest.mark.parametrize("case_id,question,must,must_not", CASES, ids=[c[0] for c in CASES])
def test_golden_case(case_id, question, must, must_not):
    answers = ask_bot(case_id, question)
    passed = sum(passes(a, must, must_not) for a in answers)
    assert passed >= MIN_PASSES, f"{passed}/5 runs passed"

Writing test_golden.py


## 8. Run it on the old prompt

`-q` makes pytest print one character per test. The last line prints the exit code of the command, which is what a CI service looks at.

**Look for:** 12 dots, `12 passed` (the time may differ) and `exit code 0`.

In [8]:
!PROMPT_VERSION=v1 python -m pytest -q test_golden.py; echo "exit code $?"

............                                                             [100%]
12 passed in 0.01s
exit code 0


## 9. Run it on the edited prompt

The same file with `PROMPT_VERSION=v2`. This run is meant to fail: that is the regression test doing its job.

**Look for:** `..F.........`: the third test fails. The report names `test_golden_case[same-day-cancel]`, says `AssertionError: 0/5 runs passed`, ends with `1 failed, 11 passed`, and the exit code is `1`. In CI, the non-zero exit code is what blocks the merge.

In [9]:
!PROMPT_VERSION=v2 python -m pytest -q test_golden.py; echo "exit code $?"

..F.........                                                             [100%]
=================================== FAILURES ===================================
______________________ test_golden_case[same-day-cancel] _______________________

case_id = 'same-day-cancel'
question = 'I have to cancel my appointment this afternoon. Is there a fee?'
must = [['30']], must_not = []

    @pytest.mark.parametrize("case_id,question,must,must_not", CASES, ids=[c[0] for c in CASES])
    def test_golden_case(case_id, question, must, must_not):
        answers = ask_bot(case_id, question)
        passed = sum(passes(a, must, must_not) for a in answers)
>       assert passed >= MIN_PASSES, f"{passed}/5 runs passed"
E       AssertionError: 0/5 runs passed
E       assert 0 >= 4

test_golden.py:15: AssertionError
=========================== short test summary info ============================
FAILED test_golden.py::test_golden_case[same-day-cancel] - AssertionError: 0/5 runs passed
1 failed, 11 passed 

exit code 1


## Now change it

Each cell below has one line marked `EDIT THIS LINE`. Run it as it is first, then change that line and run it again.

### Experiment 1: a stricter bar

This cell copies the test file with a different `MIN_PASSES` and runs it on v2. With `5` (every run must pass), **two** tests fail: `same-day-cancel` and `child-age`. The strict rule catches the regression but also blocks on the one noisy run. Try `3`: only `same-day-cancel` fails, as with 4, because it has 0/5.

In [10]:
MIN_PASSES_TRY = 5   # EDIT THIS LINE: try 5, 4 or 3
src = open("test_golden.py").read().replace("MIN_PASSES = 4", f"MIN_PASSES = {MIN_PASSES_TRY}")
open("test_try.py", "w").write(src)
!PROMPT_VERSION=v2 python -m pytest -q test_try.py; echo "exit code $?"

..F.....F...                                                             [100%]
=================================== FAILURES ===================================
______________________ test_golden_case[same-day-cancel] _______________________

case_id = 'same-day-cancel'
question = 'I have to cancel my appointment this afternoon. Is there a fee?'
must = [['30']], must_not = []

    @pytest.mark.parametrize("case_id,question,must,must_not", CASES, ids=[c[0] for c in CASES])
    def test_golden_case(case_id, question, must, must_not):
        answers = ask_bot(case_id, question)
        passed = sum(passes(a, must, must_not) for a in answers)
>       assert passed >= MIN_PASSES, f"{passed}/5 runs passed"
E       AssertionError: 0/5 runs passed
E       assert 0 >= 5

test_try.py:15: AssertionError
_________________________ test_golden_case[child-age] __________________________

case_id = 'child-age', question = 'Can I bring my 2-year-old for a check-up?'
must = [['age 3', 'aged 3', 'from 3

exit code 1


### Experiment 2: the fixed prompt

`v3` is the honest fix: the teammate's goal, narrowed to treatment prices, with the fees allowed. With `v3`, all 12 tests pass and the exit code is `0`: the fix is proved with the same test. Change it back to `v2` to see the failure again.

In [11]:
VERSION = "v3"   # EDIT THIS LINE: try "v1", "v2" or "v3"
!PROMPT_VERSION={VERSION} python -m pytest -q test_golden.py; echo "exit code $?"

............                                                             [100%]
12 passed in 0.01s
exit code 0


### Experiment 3: the dishonest fix

What if you change the check instead of the prompt? This cell grades `same-day-cancel` on v2 with a looser `must` list. With `["30", "fee"]` (accept any answer that mentions a fee) the case passes 5/5, but none of the answers says 30: the test is green and patients still do not learn the amount. Set it back to `["30"]` and it is 0/5 again.

In [12]:
LOOSE_MUST = [["30", "fee"]]   # EDIT THIS LINE: try [["30"]]
runs = answers["v2"]["same-day-cancel"]
print("passed:", sum(passes(a, LOOSE_MUST, []) for a in runs), "/ 5")
print("answers that contain 30:", sum("30" in a for a in runs), "/ 5")

passed: 5 / 5
answers that contain 30: 0 / 5


## What you should see

- With `PROMPT_VERSION=v1`, pytest prints "12 passed" (the time may differ) and the exit code is 0.
- With `v2`, exactly one test fails: `test_golden_case[same-day-cancel]`, with "0/5 runs passed".
- With `v2` and `MIN_PASSES = 5` (experiment 1), 2 tests fail: `same-day-cancel` and `child-age`.
- With `v3` (the fixed line, experiment 2), all 12 tests pass again.